# **1. Perkenalan Dataset**

## German Credit Risk Dataset

**Sumber**: UCI Machine Learning Repository (via OpenML)  
**ID OpenML**: credit-g (ID: 31)  

### Deskripsi Dataset
Dataset ini berisi informasi mengenai profil kredit nasabah dari sebuah bank di Jerman. Setiap entri merepresentasikan seorang nasabah yang diklasifikasikan sebagai risiko kredit **baik (good)** atau **buruk (bad)**.

### Karakteristik Dataset
- **Jumlah Sampel**: 1000
- **Jumlah Fitur**: 20 (7 numerik, 13 kategorikal)
- **Target Variable**: `class` (good/bad)
- **Distribusi Target**: 700 good (70%), 300 bad (30%) — imbalanced

### Fitur-fitur Utama
| No | Fitur | Tipe | Deskripsi |
|----|-------|------|-----------|
| 1 | checking_status | Kategorikal | Status rekening koran |
| 2 | duration | Numerik | Durasi kredit (bulan) |
| 3 | credit_history | Kategorikal | Riwayat kredit |
| 4 | purpose | Kategorikal | Tujuan kredit |
| 5 | credit_amount | Numerik | Jumlah kredit |
| 6 | savings_status | Kategorikal | Status tabungan |
| 7 | employment | Kategorikal | Lama bekerja |
| 8 | installment_commitment | Numerik | Persentase angsuran |
| 9 | personal_status | Kategorikal | Status personal & gender |
| 10 | other_parties | Kategorikal | Penjamin lain |
| 11 | residence_since | Numerik | Lama tinggal |
| 12 | property_magnitude | Kategorikal | Jenis properti |
| 13 | age | Numerik | Usia |
| 14 | other_payment_plans | Kategorikal | Rencana pembayaran lain |
| 15 | housing | Kategorikal | Status kepemilikan rumah |
| 16 | existing_credits | Numerik | Jumlah kredit yang ada |
| 17 | job | Kategorikal | Jenis pekerjaan |
| 18 | num_dependents | Numerik | Jumlah tanggungan |
| 19 | own_telephone | Kategorikal | Kepemilikan telepon |
| 20 | foreign_worker | Kategorikal | Pekerja asing |

### Tujuan Analisis
Membangun model machine learning untuk memprediksi risiko kredit nasabah (good/bad) berdasarkan profil dan riwayat keuangan mereka.

# **2. Import Library**

Pada tahap ini, kita mengimpor pustaka Python yang diperlukan untuk analisis data dan pembangunan model machine learning.

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

print("Semua library berhasil diimpor!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# **3. Memuat Dataset**

Pada tahap ini, kita memuat dataset German Credit Risk dari file CSV. Dataset ini sebelumnya telah diunduh dari OpenML dan disimpan secara lokal.

In [ ]:
# Memuat dataset
df = pd.read_csv('credit_risk_raw.csv')

# Menampilkan informasi dasar
print(f"Shape dataset: {df.shape}")
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")
print()

# Menampilkan 5 baris pertama
print("=" * 80)
print("5 BARIS PERTAMA DATASET")
print("=" * 80)
df.head()

In [ ]:
# Memeriksa tipe data dan informasi dataset
print("=" * 80)
print("INFORMASI DATASET")
print("=" * 80)
df.info()

In [ ]:
# Statistik deskriptif untuk fitur numerik
print("=" * 80)
print("STATISTIK DESKRIPTIF (NUMERIK)")
print("=" * 80)
df.describe()

In [ ]:
# Statistik deskriptif untuk fitur kategorikal
print("=" * 80)
print("STATISTIK DESKRIPTIF (KATEGORIKAL)")
print("=" * 80)
df.describe(include=['object', 'category'])

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita melakukan Exploratory Data Analysis (EDA) untuk memahami karakteristik dataset secara mendalam.

Tujuan dari EDA adalah:
1. Memahami distribusi setiap fitur
2. Mengidentifikasi missing values dan outlier
3. Menemukan korelasi antar fitur
4. Memahami hubungan fitur terhadap target variable

In [ ]:
# 4.1 Memeriksa Missing Values
print("=" * 80)
print("ANALISIS MISSING VALUES")
print("=" * 80)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage (%)': missing_pct
}).sort_values('Missing Count', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0])

if missing.sum() == 0:
    print("\nTidak ada missing values pada dataset! ✓")
else:
    print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# 4.2 Distribusi Target Variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
target_counts = df['class'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(target_counts.index, target_counts.values, color=colors, edgecolor='black', alpha=0.8)
axes[0].set_title('Distribusi Target Variable (class)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90, explode=[0, 0.05], shadow=True,
            textprops={'fontsize': 12})
axes[1].set_title('Proporsi Target Variable', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nDistribusi target:\n{target_counts}")
print(f"\nDataset bersifat IMBALANCED (70% good, 30% bad)")

In [ ]:
# 4.3 Distribusi Fitur Numerik
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
print(f"Fitur Numerik ({len(numeric_cols)}): {list(numeric_cols)}")

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Distribusi: {col}', fontsize=11, fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frekuensi')
        axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
        axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
        axes[i].legend(fontsize=8)

# Hide unused axes
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_numeric_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.4 Deteksi Outlier menggunakan Boxplot
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if i < len(axes):
        bp = axes[i].boxplot(df[col], patch_artist=True, 
                             boxprops=dict(facecolor='#3498db', alpha=0.7),
                             medianprops=dict(color='red', linewidth=2))
        axes[i].set_title(f'Boxplot: {col}', fontsize=11, fontweight='bold')
        axes[i].set_ylabel(col)
        
        # Hitung outlier
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
        axes[i].text(0.95, 0.95, f'Outliers: {outliers}', transform=axes[i].transAxes,
                     fontsize=9, verticalalignment='top', horizontalalignment='right',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Deteksi Outlier - Boxplot', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_boxplot_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.5 Correlation Heatmap (Fitur Numerik)
plt.figure(figsize=(12, 8))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap - Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.6 Analisis Fitur Kategorikal terhadap Target
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
categorical_cols = [col for col in categorical_cols if col != 'class']
print(f"Fitur Kategorikal ({len(categorical_cols)}): {list(categorical_cols)}")

fig, axes = plt.subplots(4, 3, figsize=(20, 22))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if i < len(axes):
        ct = pd.crosstab(df[col], df['class'], normalize='index') * 100
        ct.plot(kind='bar', ax=axes[i], stacked=True, color=['#2ecc71', '#e74c3c'], alpha=0.8)
        axes[i].set_title(f'{col} vs Target', fontsize=10, fontweight='bold')
        axes[i].set_xlabel('')
        axes[i].set_ylabel('Percentage (%)')
        axes[i].legend(title='class', fontsize=8)
        axes[i].tick_params(axis='x', rotation=45, labelsize=8)

for j in range(len(categorical_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Analisis Fitur Kategorikal vs Target', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4.7 Ringkasan EDA
print("=" * 80)
print("RINGKASAN EXPLORATORY DATA ANALYSIS")
print("=" * 80)
print()
print("1. MISSING VALUES:")
print(f"   - Tidak ada missing values pada dataset ✓")
print()
print("2. DISTRIBUSI TARGET:")
print(f"   - Good: 700 (70%) | Bad: 300 (30%)")
print(f"   - Dataset bersifat IMBALANCED")
print()
print("3. OUTLIER:")
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    if outliers > 0:
        print(f"   - {col}: {outliers} outlier(s)")
print()
print("4. KORELASI:")
print(f"   - credit_amount dan duration memiliki korelasi positif moderat")
print()
print("5. INSIGHT KATEGORIKAL:")
print(f"   - checking_status 'no checking' memiliki proporsi 'good' tertinggi")
print(f"   - credit_history 'critical/other existing credit' cenderung 'good'")
print(f"   - purpose 'new car' memiliki proporsi 'bad' yang relatif tinggi")

# **5. Data Preprocessing**

Pada tahap ini, kita melakukan preprocessing data untuk mempersiapkan dataset agar siap digunakan untuk pelatihan model machine learning.

Tahapan preprocessing:
1. Menangani Missing Values
2. Menghapus Data Duplikat
3. Encoding Data Kategorikal (LabelEncoder)
4. Deteksi dan Penanganan Outlier (IQR Capping)
5. Normalisasi/Standarisasi Fitur (StandardScaler)
6. Split Data (Train/Test)

In [ ]:
# 5.1 Menangani Missing Values
print("=" * 60)
print("TAHAP 1: Menangani Missing Values")
print("=" * 60)

df_clean = df.copy()

# Cek missing values
missing = df_clean.isnull().sum()
print(f"Missing values per kolom:")
print(missing[missing > 0] if missing.sum() > 0 else "Tidak ada missing values ✓")

# Jika ada missing values, isi dengan median (numerik) atau modus (kategorikal)
for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['float64', 'int64']:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
        else:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print(f"\nMissing values setelah penanganan: {df_clean.isnull().sum().sum()}")

In [ ]:
# 5.2 Menghapus Data Duplikat
print("=" * 60)
print("TAHAP 2: Menghapus Data Duplikat")
print("=" * 60)

duplicates = df_clean.duplicated().sum()
print(f"Jumlah data duplikat: {duplicates}")

df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Shape setelah menghapus duplikat: {df_clean.shape}")

In [ ]:
# 5.3 Encoding Data Kategorikal
print("=" * 60)
print("TAHAP 3: Encoding Data Kategorikal")
print("=" * 60)

label_encoders = {}
categorical_cols = df_clean.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded '{col}': {len(le.classes_)} classes -> {list(le.classes_)[:5]}...")

print(f"\nTotal kolom yang di-encode: {len(label_encoders)}")
print(f"\nSample data setelah encoding:")
df_clean.head()

In [ ]:
# 5.4 Deteksi dan Penanganan Outlier (IQR Capping)
print("=" * 60)
print("TAHAP 4: Penanganan Outlier (IQR Capping)")
print("=" * 60)

numeric_cols_clean = df_clean.select_dtypes(include=['float64', 'int64']).columns
numeric_cols_clean = [col for col in numeric_cols_clean if col != 'class']

for col in numeric_cols_clean:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = ((df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)).sum()
    
    if outlier_count > 0:
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
        print(f"  {col}: {outlier_count} outlier(s) di-capping ke [{lower_bound:.2f}, {upper_bound:.2f}]")

print("\nPenanganan outlier selesai ✓")

In [ ]:
# 5.5 Normalisasi Fitur (StandardScaler)
print("=" * 60)
print("TAHAP 5: Normalisasi Fitur (StandardScaler)")
print("=" * 60)

feature_cols = [col for col in df_clean.columns if col != 'class']
scaler = StandardScaler()
df_clean[feature_cols] = scaler.fit_transform(df_clean[feature_cols])

print(f"Scaled {len(feature_cols)} features")
print(f"\nStatistik setelah scaling:")
df_clean[feature_cols].describe().round(3)

In [ ]:
# 5.6 Split Data (Train/Test)
print("=" * 60)
print("TAHAP 6: Split Data (Train 80% / Test 20%)")
print("=" * 60)

X = df_clean.drop(columns=['class'])
y = df_clean['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(df_clean)*100:.1f}%)")
print(f"Test set:  {X_test.shape[0]} samples ({X_test.shape[0]/len(df_clean)*100:.1f}%)")
print(f"\nDistribusi target (train):")
print(y_train.value_counts())
print(f"\nDistribusi target (test):")
print(y_test.value_counts())

In [ ]:
# 5.7 Menyimpan Data yang Sudah Dipreprocess
print("=" * 60)
print("MENYIMPAN DATA PREPROCESSED")
print("=" * 60)

import os
os.makedirs('preprocessing', exist_ok=True)

# Save train and test sets
train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('preprocessing/credit_risk_train.csv', index=False)
test_df.to_csv('preprocessing/credit_risk_test.csv', index=False)

# Save full preprocessed dataset
df_clean.to_csv('preprocessing/credit_risk_preprocessed.csv', index=False)

print(f"✓ Train data saved: preprocessing/credit_risk_train.csv ({train_df.shape})")
print(f"✓ Test data saved:  preprocessing/credit_risk_test.csv ({test_df.shape})")
print(f"✓ Full data saved:  preprocessing/credit_risk_preprocessed.csv ({df_clean.shape})")
print()
print("=" * 60)
print("PREPROCESSING SELESAI!")
print("=" * 60)